# fastlinkage on an NVIDIA GPU

Checks that fastlinkage is exact and fast on CUDA. In Colab choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes a few minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/michael-L-i/fastlinkage
%pip install -q -e "fastlinkage[test]" fastcluster

## 1. Test suite

Every test runs on the CPU and on the GPU, and compares the tree with SciPy or scikit-learn.

In [ ]:
!cd fastlinkage && python -m pytest -q

## 2. Exactness at 20,000 points, against SciPy

In [ ]:
import sys, time
import numpy as np, torch
from scipy.cluster.hierarchy import linkage as scipy_linkage
sys.path.insert(0, "fastlinkage/src")
from fastlinkage import METHODS, linkage

assert torch.cuda.is_available(), "no CUDA GPU: change the runtime type"
SCIPY = {"ward": ("ward", "euclidean"), "average_cosine": ("average", "cosine"),
         "average_sqeuclidean": ("average", "sqeuclidean")}

def blobs(n, d, seed=0):
    rng = np.random.default_rng(seed)
    centres = rng.normal(size=(200, d)) * 4
    return centres[rng.integers(0, 200, n)] + rng.normal(size=(n, d))

def timed(X, method, device="cuda"):
    t0 = time.perf_counter()
    Z = linkage(X, method, device=device)
    return Z, time.perf_counter() - t0

linkage(blobs(3000, 32), "ward", device="cuda")   # warm-up
report = [f"GPU: {torch.cuda.get_device_name(0)}, torch {torch.__version__}"]
X = blobs(20_000, 32)
for method in METHODS:
    Z, t = timed(X, method)
    t0 = time.perf_counter(); Zs = scipy_linkage(X, *SCIPY[method]); ts = time.perf_counter() - t0
    diff = float(np.max(np.abs(Z[:, 2] - Zs[:, 2])))
    report.append(f"20,000 x 32   {method:20s} cuda {t:6.2f} s   scipy {ts:6.1f} s   max height diff {diff:.1e}")
    print(report[-1])

## 3. Timings at larger sizes

fastcluster's exact Ward is run once at 100,000 points as an independent check (about two minutes).

In [ ]:
import fastcluster

for n, d in ((100_000, 32), (100_000, 384), (200_000, 32)):
    X = blobs(n, d)
    for method in METHODS:
        Z, t = timed(X, method)
        line = f"{n:,} x {d:<4d} {method:20s} cuda {t:6.2f} s"
        if (n, d, method) == (100_000, 32, "ward"):
            t0 = time.perf_counter(); Zf = fastcluster.linkage_vector(X, method="ward"); tf = time.perf_counter() - t0
            line += f"   fastcluster {tf:6.1f} s   max height diff {float(np.max(np.abs(Z[:, 2] - Zf[:, 2]))):.1e}"
        report.append(line)
        print(line)
    torch.cuda.empty_cache()

## 4. Summary

Copy this block into a GitHub issue or the README.

In [ ]:
print("\n".join(report))